# Surface plasmon at a dielectric-metal interface

A surface plasmon polariton (SPP) is a lossy TM mode bound to the interface between a dielectric and a metal. This example uses REMSOL's literal two-layer interface solver to compute its complex effective index and field, then compares the result with the closed-form planar-interface dispersion.

REMSOL's `omega` argument is `k0 = 2*pi/wavelength` in rad/um. The real part of `neff` sets the phase constant; its positive imaginary part gives attenuation along the propagation direction.

The metal model is a damped Drude approximation using representative gold-like parameters. It follows the free-electron term discussed by A. D. Rakic et al., [Applied Optics 37, 5271-5283 (1998)](https://doi.org/10.1364/AO.37.005271), but omits the Lorentz/interband oscillators from their full model. The results are therefore a solver demonstration, not an experimental model of a particular gold film.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import remsol
from remsol import Polarization as pol

EPSILON_D = 1.45**2
EPSILON_INF = 9.5
E_P = 9.03  # eV
GAMMA = 0.071  # eV
HC = 1.239841984  # eV um
WAVELENGTHS = np.linspace(0.8, 2.0, 31)

## Material model

The Drude permittivity is evaluated in photon-energy units. `passive_sqrt` selects the refractive-index branch with nonnegative imaginary part, as required for a passive metal. The first and last layer thicknesses set only the two plotting windows for this exact interface.

In [ ]:
def gold_permittivity(wavelength):
    energy = HC / np.asarray(wavelength)
    return EPSILON_INF - E_P**2 / (energy * (energy + 1j * GAMMA))


def passive_sqrt(value):
    root = complex(np.sqrt(value))
    return -root if root.imag < 0.0 else root


def make_interface(wavelength):
    epsilon_m = complex(gold_permittivity(wavelength))
    return remsol.MultiLayer(
        [
            remsol.Layer(n=1.45, d=2.0),
            remsol.Layer(n=passive_sqrt(epsilon_m), d=2.0),
        ]
    )


def analytic_spp_neff(epsilon_m):
    candidate = complex(
        np.sqrt(EPSILON_D * epsilon_m / (EPSILON_D + epsilon_m))
    )
    if candidate.real < 0.0 or (
        abs(candidate.real) < 1e-15 and candidate.imag < 0.0
    ):
        candidate = -candidate
    return candidate

In [ ]:
epsilon_m = gold_permittivity(WAVELENGTHS)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(WAVELENGTHS, epsilon_m.real, label="Re(epsilon_m)")
ax.plot(WAVELENGTHS, epsilon_m.imag, label="Im(epsilon_m)")
ax.axhline(-EPSILON_D, color="black", linestyle="--", label="-epsilon_d")
ax.set(xlabel="Wavelength (um)", ylabel="Relative permittivity")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Complex dispersion

For a two-layer structure, `complex_neff` evaluates the exact unsquared, branch-aware interface condition. No search rectangle is needed. The analytic comparison uses the squared dispersion only to form a candidate, with the same positive-propagation convention.

In [ ]:
remsol_neff = []
analytic_neff = []

for wavelength, metal_epsilon in zip(WAVELENGTHS, epsilon_m):
    omega = 2.0 * np.pi / wavelength
    result = make_interface(wavelength).complex_neff(omega, pol.TM)
    if result is None:
        raise RuntimeError(f"No TM interface mode at wavelength={wavelength:.3f} um")
    remsol_neff.append(complex(*result))
    analytic_neff.append(analytic_spp_neff(complex(metal_epsilon)))

remsol_neff = np.asarray(remsol_neff)
analytic_neff = np.asarray(analytic_neff)
max_error = np.max(np.abs(remsol_neff - analytic_neff))
print(f"Maximum |neff_REMSOL - neff_analytic|: {max_error:.3e}")
assert max_error < 1e-8
assert np.all(remsol_neff.imag > 0.0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

axes[0].plot(WAVELENGTHS, analytic_neff.real, "k-", label="analytic")
axes[0].plot(WAVELENGTHS, remsol_neff.real, "o", ms=4, label="REMSOL")
axes[0].set_ylabel("Re(neff)")

axes[1].plot(WAVELENGTHS, analytic_neff.imag, "k-", label="analytic")
axes[1].plot(WAVELENGTHS, remsol_neff.imag, "o", ms=4, label="REMSOL")
axes[1].set_ylabel("Im(neff)")

for ax in axes:
    ax.set_xlabel("Wavelength (um)")
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

## Propagation length

With `beta = k0*neff`, intensity decays as `exp(-2*Im(beta)*z)`. The 1/e intensity propagation length is therefore `1/(2*Im(beta))`.

In [ ]:
k0 = 2.0 * np.pi / WAVELENGTHS
beta = k0 * remsol_neff
propagation_length = 1.0 / (2.0 * beta.imag)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(WAVELENGTHS, propagation_length)
ax.set(
    xlabel="Wavelength (um)",
    ylabel="1/e intensity propagation length (um)",
)
ax.grid(alpha=0.3)
plt.show()

## TM field at 1.55 um

The interface is at `x=0`: dielectric on the left and metal on the right. MaxField normalization makes the largest total electric-field magnitude equal to one.

In [ ]:
wavelength = 1.55
omega = 2.0 * np.pi / wavelength
interface = make_interface(wavelength)
field = interface.complex_field(omega, pol.TM)

x = np.asarray(field.x)
ex = np.asarray(field.Ex)
ez = np.asarray(field.Ez)
hy = np.asarray(field.Hy)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, np.abs(ex), label="|Ex|")
ax.plot(x, np.abs(ez), label="|Ez|")
ax.plot(x, np.abs(hy), label="|Hy|")
ax.axvline(0.0, color="black", linewidth=1.0)
ax.axvspan(0.0, x.max(), color="goldenrod", alpha=0.15, label="Drude metal")
ax.set(xlabel="Transverse coordinate x (um)", ylabel="Normalized amplitude")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

In [ ]:
interface_index = int(np.flatnonzero(x >= 0.0)[0])
left = interface_index - 1
right = interface_index
epsilon_m_1550 = complex(gold_permittivity(wavelength))

max_e = np.max(np.sqrt(np.abs(ex)**2 + np.abs(ez)**2))
hy_jump = abs(hy[left] - hy[right]) / max(
    abs(hy[left]) + abs(hy[right]), np.finfo(float).tiny
)
dz_left = EPSILON_D * ez[left]
dz_right = epsilon_m_1550 * ez[right]
dz_jump = abs(dz_left - dz_right) / max(
    abs(dz_left) + abs(dz_right), np.finfo(float).tiny
)

print(f"epsilon_m(1.55 um) = {epsilon_m_1550:.6g}")
print(f"neff(1.55 um) = {interface.complex_neff(omega, pol.TM)}")
print(f"max |E| = {max_e:.12f}")
print(f"relative Hy jump across sampled interface = {hy_jump:.3e}")
print(f"relative epsilon*Ez jump across sampled interface = {dz_jump:.3e}")

assert np.isclose(max_e, 1.0)
assert hy_jump < 1e-2
assert dz_jump < 1e-2
assert np.abs(ex[0]) + np.abs(ez[0]) < 0.3
assert np.abs(ex[-1]) + np.abs(ez[-1]) < 0.3

The mode is evanescent on both sides and strongly confined at the interface. `Hy` and `epsilon*Ez` are continuous; `Ez` itself jumps because the permittivity changes.

The damped Drude model is intentionally simple. It omits explicit Lorentz/interband oscillators and should not be treated as a fit to a specific deposited film. REMSOL also models a one-dimensional planar interface and does not include surface roughness, finite lateral extent, or nonlinear effects.